# 02 - Generate Chain-of-Thought Responses

Uses Qwen3-4B's native thinking mode to generate COT for each GSM8K problem.

**Fully resumable** - caches one JSON file per problem in `cache/cots/`. Re-run all cells to continue from where you left off.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import re
import traceback
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from lib.data import load_gsm8k, extract_predicted_answer, build_generation_messages

In [ ]:
# Load model for generation (raw transformers -- we need .generate() which nnsight doesn't wrap)
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16
)
model.eval()
print(f"Loaded on {next(model.parameters()).device}")

In [ ]:
# Load dataset
dataset = load_gsm8k()
print(f"Total GSM8K problems: {len(dataset)}")

# Resume logic
done_ids = {int(p.stem) for p in COT_CACHE.glob("*.json")}
todo = [ex for ex in dataset if ex["problem_id"] not in done_ids]
print(f"Resuming: {len(done_ids)} done, {len(todo)} remaining")

In [ ]:
def generate_cot(question: str) -> dict:
    """Generate COT response for a single question using Qwen3 thinking mode.
    
    Returns dict with 'full_response', 'cot_text', 'predicted_answer'.
    """
    messages = build_generation_messages(question)
    
    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=MAX_COT_TOKENS,
            temperature=TEMPERATURE,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    
    # Decode only the generated tokens
    generated = tokenizer.decode(output_ids[0, input_ids.shape[1]:], skip_special_tokens=False)
    
    # Extract thinking content from <think>...</think> tags
    think_match = re.search(r"<think>(.*?)</think>", generated, re.DOTALL)
    cot_text = think_match.group(1).strip() if think_match else generated.strip()
    
    # Extract the answer portion (after </think>)
    if think_match:
        answer_portion = generated[think_match.end():].strip()
    else:
        answer_portion = generated.strip()
    
    predicted_answer = extract_predicted_answer(answer_portion)
    
    return {
        "full_response": generated,
        "cot_text": cot_text,
        "answer_portion": answer_portion,
        "predicted_answer": predicted_answer,
    }

In [ ]:
# Generate COTs with per-problem caching
correct = 0
total = 0

for ex in tqdm(todo, desc="Generating COTs"):
    try:
        result = generate_cot(ex["question"])
        cache_entry = {
            "problem_id": ex["problem_id"],
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "cot_text": result["cot_text"],
            "full_response": result["full_response"],
            "answer_portion": result["answer_portion"],
            "predicted_answer": result["predicted_answer"],
            "error": None,
        }
        if result["predicted_answer"] == ex["gold_answer"]:
            correct += 1
        total += 1
    except Exception:
        cache_entry = {
            "problem_id": ex["problem_id"],
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "cot_text": None,
            "full_response": None,
            "answer_portion": None,
            "predicted_answer": None,
            "error": traceback.format_exc(),
        }
    
    (COT_CACHE / f"{ex['problem_id']}.json").write_text(json.dumps(cache_entry))

if total > 0:
    print(f"\nBatch accuracy: {correct}/{total} = {correct/total:.1%}")

In [ ]:
# Aggregate all cache files into results/cots.jsonl
all_cots = []
errors = 0
correct = 0

for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    entry = json.loads(p.read_text())
    all_cots.append(entry)
    if entry["error"]:
        errors += 1
    elif entry["predicted_answer"] == entry["gold_answer"]:
        correct += 1

# Write aggregated results
output_path = RESULTS_DIR / "cots.jsonl"
with open(output_path, "w") as f:
    for entry in all_cots:
        f.write(json.dumps(entry) + "\n")

valid = len(all_cots) - errors
print(f"Total: {len(all_cots)} | Valid: {valid} | Errors: {errors}")
print(f"Normal accuracy (ceiling): {correct}/{valid} = {correct/valid:.1%}" if valid > 0 else "No valid results")
print(f"Saved to {output_path}")

In [ ]:
# Show a few examples
for entry in all_cots[:3]:
    if entry["error"]:
        continue
    print(f"--- Problem {entry['problem_id']} ---")
    print(f"Q: {entry['question'][:100]}...")
    print(f"COT (first 200 chars): {entry['cot_text'][:200]}...")
    print(f"Predicted: {entry['predicted_answer']} | Gold: {entry['gold_answer']}")
    print()